##### Copyright 2025 Google LLC.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.


# cvrptw_break

<table align="left">
<td>
<a href="https://colab.research.google.com/github/google/or-tools/blob/main/examples/notebook/routing/cvrptw_break.ipynb"><img src="https://raw.githubusercontent.com/google/or-tools/main/tools/colab_32px.png"/>Run in Google Colab</a>
</td>
<td>
<a href="https://github.com/google/or-tools/blob/main/ortools/routing/samples/cvrptw_break.py"><img src="https://raw.githubusercontent.com/google/or-tools/main/tools/github_32px.png"/>View source on GitHub</a>
</td>
</table>

First, you must install [ortools](https://pypi.org/project/ortools/) package in this colab.

In [ ]:
%pip install ortools


Capacitated Vehicle Routing Problem with Time Windows (CVRPTW).

This is a sample using the routing library python wrapper to solve a CVRPTW
problem.
A description of the problem can be found here:
http://en.wikipedia.org/wiki/Vehicle_routing_problem.

Distances are in meters and time in minutes.



In [ ]:
import functools
from typing import Any, Dict

from ortools.constraint_solver.python import constraint_solver
from ortools.routing import enums_pb2, parameters_pb2
from ortools.routing.python import routing



def create_data_model() -> Dict[str, Any]:
    """Stores the data for the problem."""
    data = {}
    # Locations in block unit
    locations_ = [
        # fmt: off
      (4, 4),  # depot
      (2, 0), (8, 0),  # locations to visit
      (0, 1), (1, 1),
      (5, 2), (7, 2),
      (3, 3), (6, 3),
      (5, 5), (8, 5),
      (1, 6), (2, 6),
      (3, 7), (6, 7),
      (0, 8), (7, 8),
        # fmt: on
    ]
    # Compute locations in meters using the block dimension defined as follow
    # Manhattan average block: 750ft x 264ft -> 228m x 80m
    # here we use: 114m x 80m city block
    # src: https://nyti.ms/2GDoRIe "NY Times: Know Your distance"
    data["locations"] = [(l[0] * 114, l[1] * 80) for l in locations_]
    data["numlocations_"] = len(data["locations"])
    data["time_windows"] = [
        # fmt: off
      (0, 0),  # depot
      (75, 85), (75, 85),  #  1,  2
      (60, 70), (45, 55),  #  3,  4
      (0, 8), (50, 60),    #  5,  6
      (0, 10), (10, 20),   #  7,  8
      (0, 10), (75, 85),   #  9, 10
      (85, 95), (5, 15),   # 11, 12
      (15, 25), (10, 20),  # 13, 14
      (45, 55), (30, 40),
        # 15, 16
        # fmt: on
    ]
    data["demands"] = [
        # fmt: off
      0,     # depot
      1, 1,  #  1,  2
      2, 4,  #  3,  4
      2, 4,  #  5,  6
      8, 8,  #  7,  8
      1, 2,  #  9, 10
      1, 2,  # 11, 12
      4, 4,  # 13, 14
      8, 8,
        # 15, 16
        # fmt: on
    ]
    data["time_per_demand_unit"] = 5  # 5 minutes/unit
    data["num_vehicles"] = 4
    data["breaks"] = [(2, False), (2, False), (2, False), (2, False)]
    data["vehicle_capacity"] = 15
    data["vehicle_speed"] = 83  # Travel speed: 5km/h converted in m/min
    data["depot"] = 0
    return data


def manhattan_distance(position_1: tuple[int, int], position_2: tuple[int, int]) -> int:
    """Computes the Manhattan distance between two points."""
    return abs(position_1[0] - position_2[0]) + abs(position_1[1] - position_2[1])


def create_distance_evaluator(data: Dict[str, Any]) -> Any:
    """Creates callback to return distance between points."""
    distances_ = {}
    # precompute distance between location to have distance callback in O(1)
    for from_node in range(data["numlocations_"]):
        distances_[from_node] = {}
        for to_node in range(data["numlocations_"]):
            if from_node == to_node:
                distances_[from_node][to_node] = 0
            else:
                distances_[from_node][to_node] = manhattan_distance(
                    data["locations"][from_node], data["locations"][to_node]
                )

    def distance_evaluator(
        manager: routing.IndexManager, from_node: int, to_node: int
    ) -> int:
        """Returns the manhattan distance between the two nodes."""
        return distances_[manager.index_to_node(from_node)][
            manager.index_to_node(to_node)
        ]

    return distance_evaluator


def create_demand_evaluator(data: Dict[str, Any]) -> Any:
    """Creates callback to get demands at each location."""
    demands_ = data["demands"]

    def demand_evaluator(manager: routing.IndexManager, node: int) -> int:
        """Returns the demand of the current node."""
        return demands_[manager.index_to_node(node)]

    return demand_evaluator


def add_capacity_constraints(
    routing_model: routing.Model,
    data: Dict[str, Any],
    demand_evaluator_index: int,
) -> None:
    """Adds capacity constraint."""
    capacity = "Capacity"
    routing_model.add_dimension(
        demand_evaluator_index,
        0,  # null capacity slack
        data["vehicle_capacity"],
        True,  # start cumul to zero
        capacity,
    )


def create_time_evaluator(data: Dict[str, Any]) -> Any:
    """Creates callback to get total times between locations."""

    def service_time(data: Dict[str, Any], node: int) -> int:
        """Gets the service time for the specified location."""
        return data["demands"][node] * data["time_per_demand_unit"]

    def travel_time(data: Dict[str, Any], from_node: int, to_node: int) -> int:
        """Gets the travel times between two locations."""
        if from_node == to_node:
            travel_time = 0
        else:
            travel_time = (
                manhattan_distance(
                    data["locations"][from_node], data["locations"][to_node]
                )
                / data["vehicle_speed"]
            )
        return int(travel_time)

    total_time_ = {}
    # precompute total time to have time callback in O(1)
    for from_node in range(data["numlocations_"]):
        total_time_[from_node] = {}
        for to_node in range(data["numlocations_"]):
            if from_node == to_node:
                total_time_[from_node][to_node] = 0
            else:
                total_time_[from_node][to_node] = int(
                    service_time(data, from_node)
                    + travel_time(data, from_node, to_node)
                )

    def time_evaluator(
        manager: routing.IndexManager, from_node: int, to_node: int
    ) -> int:
        """Returns the total time between the two nodes."""
        return total_time_[manager.index_to_node(from_node)][
            manager.index_to_node(to_node)
        ]

    return time_evaluator


def add_time_window_constraints(
    routing_model: routing.Model,
    manager: routing.IndexManager,
    data: Dict[str, Any],
    time_evaluator_index: int,
) -> None:
    """Add Global Span constraint."""
    time = "Time"
    horizon = 120
    routing_model.add_dimension(
        time_evaluator_index,
        horizon,  # allow waiting time
        horizon,  # maximum time per vehicle
        False,  # don't force start cumul to zero
        time,
    )
    time_dimension = routing_model.get_dimension_or_die(time)
    # Add time window constraints for each location except depot
    # and 'copy' the slack var in the solution object (aka Assignment) to print it
    for location_idx, time_window in enumerate(data["time_windows"]):
        if location_idx == data["depot"]:
            continue
        index = manager.node_to_index(location_idx)
        time_dimension.cumul_var(index).set_range(time_window[0], time_window[1])
        # routing.AddToAssignment(time_dimension.SlackVar(index))
    # Add time window constraints for each vehicle start node
    # and 'copy' the slack var in the solution object (aka Assignment) to print it
    for vehicle_id in range(data["num_vehicles"]):
        index = routing_model.start(vehicle_id)
        time_dimension.cumul_var(index).set_range(
            data["time_windows"][0][0], data["time_windows"][0][1]
        )
        # routing.AddToAssignment(time_dimension.SlackVar(index))
        # The time window at the end node was impliclty set in the time dimension
        # definition to be [0, horizon].
        # Warning: Slack var is not defined for vehicle end nodes and should not
        # be added to the assignment.


def print_solution(
    data: Dict[str, Any],
    manager: routing.IndexManager,
    routing_model: routing.Model,
    assignment: constraint_solver.Assignment,
) -> None:  # pylint:disable=too-many-locals
    """Prints assignment on console."""
    print(f"Objective: {assignment.objective_value()}")

    print("Breaks:")
    intervals = assignment.interval_var_container()
    for i in range(intervals.size()):
        brk = intervals.element(i)
        if brk.performed_value() == 1:
            print(
                f"{brk.var().name}:"
                f" Start({brk.start_value()}) Duration({brk.duration_value()})"
            )
        else:
            print(f"{brk.var().name}: Unperformed")

    total_distance = 0
    total_load = 0
    total_time = 0
    capacity_dimension = routing_model.get_dimension_or_die("Capacity")
    time_dimension = routing_model.get_dimension_or_die("Time")
    for vehicle_id in range(data["num_vehicles"]):
        if not routing_model.is_vehicle_used(assignment, vehicle_id):
            continue
        index = routing_model.start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        distance = 0
        while not routing_model.is_end(index):
            load_var = capacity_dimension.cumul_var(index)
            time_var = time_dimension.cumul_var(index)
            slack_var = time_dimension.slack_var(index)
            node = manager.index_to_node(index)
            plan_output += (
                f" {node}"
                f" Load({assignment.value(load_var)})"
                f" Time({assignment.min(time_var)}, {assignment.max(time_var)})"
                f" Slack({assignment.min(slack_var)}, {assignment.max(slack_var)})"
                " ->"
            )
            previous_index = index
            index = assignment.value(routing_model.next_var(index))
            distance += routing_model.get_arc_cost_for_vehicle(
                previous_index, index, vehicle_id
            )
        load_var = capacity_dimension.cumul_var(index)
        time_var = time_dimension.cumul_var(index)
        node = manager.index_to_node(index)
        plan_output += (
            f" {node}"
            f" Load({assignment.value(load_var)})"
            f" Time({assignment.min(time_var)}, {assignment.max(time_var)})\n"
        )
        plan_output += f"Distance of the route: {distance}m\n"
        plan_output += f"Load of the route: {assignment.value(load_var)}\n"
        plan_output += f"Time of the route: {assignment.value(time_var)}\n"
        print(plan_output)
        total_distance += distance
        total_load += assignment.value(load_var)
        total_time += assignment.value(time_var)
    print(f"Total Distance of all routes: {total_distance}m")
    print(f"Total Load of all routes: {total_load}")
    print(f"Total Time of all routes: {total_time}min")


def main() -> None:
    """Entry point of the program."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager
    manager = routing.IndexManager(
        data["numlocations_"], data["num_vehicles"], data["depot"]
    )

    # Create Routing Model
    routing_model = routing.Model(manager)

    # Define weight of each edge
    distance_evaluator_index = routing_model.register_transit_callback(
        functools.partial(create_distance_evaluator(data), manager)
    )
    routing_model.set_arc_cost_evaluator_of_all_vehicles(distance_evaluator_index)

    # Add Capacity constraint
    demand_evaluator_index = routing_model.register_unary_transit_callback(
        functools.partial(create_demand_evaluator(data), manager)
    )
    add_capacity_constraints(routing_model, data, demand_evaluator_index)

    # Add Time Window constraint
    time_evaluator_index = routing_model.register_transit_callback(
        functools.partial(create_time_evaluator(data), manager)
    )
    add_time_window_constraints(routing_model, manager, data, time_evaluator_index)

    # Add breaks
    time_dimension = routing_model.get_dimension_or_die("Time")
    node_visit_transit = {}
    for index in range(routing_model.size()):
        # Add slack var to the assignment to print it.
        routing_model.add_to_assignment(time_dimension.slack_var(index))

        # Add transit for the break interval var
        node = manager.index_to_node(index)
        node_visit_transit[index] = int(
            data["demands"][node] * data["time_per_demand_unit"]
        )

    break_intervals = {}
    for v in range(data["num_vehicles"]):
        vehicle_break = data["breaks"][v]
        break_intervals[v] = [
            routing_model.solver.new_fixed_duration_interval_var(
                15,
                100,
                vehicle_break[0],
                vehicle_break[1],
                f"Break for vehicle {v}",
            )
        ]
        time_dimension.set_break_intervals_of_vehicle(
            break_intervals[v], v, list(node_visit_transit.values())
        )

    # Setting first solution heuristic (cheapest addition).
    search_parameters: parameters_pb2.RoutingSearchParameters = (
        routing.default_routing_search_parameters()
    )
    search_parameters.first_solution_strategy = (
        enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )  # pylint: disable=no-member

    # Solve the problem.
    assignment = routing_model.solve_with_parameters(search_parameters)

    # Print solution on console.
    if assignment:
        print_solution(data, manager, routing_model, assignment)
    else:
        print("No solution found!")


main()

